# Kulturpool hackathon 2026

API Call -> not as reliable, so downloaded the entire dataset 

aggregatedCHO -> Kulturpool wrapper for the cultural heritage info 

In [39]:
# imports 
from json import load
from urllib.parse import urlencode
from urllib.request import urlopen
import json
import tarfile
from pathlib import Path
from os import mkdir
from os.path import isdir
import requests
import pandas as pd 

## API Call 

In [ ]:
KULTURPOOL_SEARCH_API = "https://api.kulturpool.at/v2/"

In [10]:
def search_kulturpool():
    url = f"https://api.kulturpool.at/v2/search?q=Papier&query_by=medium"
    with urlopen(url, timeout=30) as response:
        return load(response)

response = search_kulturpool()

results = response.get("hits", response.get("results", []))
for item in results[:5]:
    print(item)

{'document': {'contentHash': '1b9561b4775a4e4c', 'dataProvider': 'Golling, Museum Burg Golling', 'date': ['1589'], 'dateMax': -11991715200, 'dateMin': -12023164800, 'dcRights': ['Salzburger Regionalmuseen und Sammlungen'], 'dcType': ['Graphik'], 'description': ['Abraham Ortelius (1527-1598), dessen Vorfahren aus Augsburg stammten, hatte eine umfassende Ausbildung genossen und arbeitete als Geograph und Kartograf in Antwerpen. Sein wohl bedeutendstes Werk, das „Theatrum Orbis Terrarum“ erschien erstmals 1570 und wurde bis 1612 in mehreren Sprachen und über vierzig Auflagen gedruckt.  \n\nDas Besondere daran: Es handelte sich dabei erstmals um eine Landkartensammlung mit erläuternden Texten in Buchform, also einen klassischen „Atlas“. Diese Bezeichnung wurde jedoch erst etliche Jahre später von Ortelius´ flämischem Berufskollegen Gerhard Mercator geprägt.  \n\nLandkarten, und nun vor allem Atlanten, avancierten bereits bald zu beliebten Sammelobjekten, die jedoch, aufgrund ihrer exorbita

## Download Dataset


In [40]:
DATASET_URL = "https://s3.kpool.at/nightly/records_export_full.tar.gz"
ARCHIVE_PATH = Path("full_records.tar.gz")

In [42]:
# von Kulturpool 

def download(url: str, target: Path) -> Path:
    """Archiv streamend herunterladen (mehrere hundert MB)."""
    if target.exists():
        print(f"{target} existiert bereits — Download übersprungen.")
        return target

    print(f"Lade Datenset von {url} herunter ...")
    with requests.get(url, stream=True, timeout=60) as response:
        response.raise_for_status()
        with target.open("wb") as fh:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                fh.write(chunk)

    print(f"Fertig: {target} ({target.stat().st_size / 1024**2:.0f} MB)")
    return target


def iter_records(archive: Path):
    """Alle Datensätze aus allen .jsonl-Dateien im Archiv liefern."""
    with tarfile.open(archive, mode="r:gz") as tar:
        for member in tar:
            if not (member.isfile() and member.name.endswith(".jsonl")):
                continue

            print(f"\n--- Verarbeite Datei: {member.name} ---")
            handle = tar.extractfile(member)
            if handle is None:
                continue

            for line_number, raw_line in enumerate(handle, 1):
                line = raw_line.decode("utf-8").strip()
                if not line:
                    continue
                try:
                    yield member.name, json.loads(line)
                except json.JSONDecodeError:
                    print(f"Fehler beim Parsen von {member.name}:{line_number}")


def get_title(record: dict) -> str | None:
    """Titel extrahieren, bevorzugt auf Deutsch."""
    titles = record.get("aggregatedCHO", {}).get("dc:title") or []
    if not titles:
        return None

    german = next((t["value"] for t in titles if t.get("lang") == "de"), None)
    return german or titles[0].get("value")


if __name__ == "__main__":
    download(DATASET_URL, ARCHIVE_PATH)

    count = 0
    for source_file, record in iter_records(ARCHIVE_PATH):
        title = get_title(record)
        if title:
            count += 1
            if count <= 10:
                print(f"Gefundener Titel: {title}")

    print(f"\nVerarbeitung abgeschlossen. {count} Datensätze mit Titel gefunden.")

full_records.tar.gz existiert bereits — Download übersprungen.

--- Verarbeite Datei: akademie-der-bildenden-kuenste-core__2026-01-14T11-50-22Z.jsonl ---

--- Verarbeite Datei: alban-berg__2025-06-17T08-09-23.305000Z.jsonl ---

--- Verarbeite Datei: albertina__2026-07-13T14-03-47Z.jsonl ---

--- Verarbeite Datei: arbeiterzeitung-archiv__2026-09-16T07-29-47Z.jsonl ---

--- Verarbeite Datei: ars-electronica__2026-06-17T10-13-21Z.jsonl ---

--- Verarbeite Datei: azw-core__2026-06-03T08-53-03Z.jsonl ---

--- Verarbeite Datei: basis-wien-moderne-kunst-roth__2026-04-10T07-24-51Z.jsonl ---

--- Verarbeite Datei: belvedere-bibliothek__2026-01-13T16-46-43Z.jsonl ---

--- Verarbeite Datei: belvedere__2025-09-17T06-47-57Z.jsonl ---

--- Verarbeite Datei: bischof-rudigier-stiftung-core__2025-11-17T11-44-21Z.jsonl ---

--- Verarbeite Datei: boku-larchiv-core__2026-08-24T15-58-52Z.jsonl ---

--- Verarbeite Datei: brenner-forum-schroedinger__2025-11-14T11-53-03Z.jsonl ---

--- Verarbeite Datei: brenn

In [44]:
# unzip the full_records.tar.gz file and extract the .jsonl files to a directory called entries 
# Source - https://stackoverflow.com/a/72484186
# Posted by Hafizur Rahman, modified by community. See post 'Timeline' for change history
# Retrieved 2026-10-27, License - CC BY-SA 4.0

src_path = 'full_records.tar.gz'
dst_path = 'outputs'

if src_path.endswith('tar.gz'):
    tar = tarfile.open(src_path, 'r:gz')
    tar.extractall(dst_path)
    tar.close()


C:\Users\lantz\AppData\Local\Temp\ipykernel_36976\2607694796.py:11: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(dst_path)


### Analysis of Dataset

In [55]:
# Percentage of objects with dcType, medium, and both fields
def calculate_percentages(records):
    total_records = 0
    dctype_count = 0
    medium_count = 0
    both_count = 0
    subject_count = 0
    invalid_json_lines = 0

    for record in records:
        if record is None:
            invalid_json_lines += 1
            continue
        total_records += 1
        aggregated_cho = record.get('content', {}).get('aggregatedCHO', {})
        has_dctype = bool(aggregated_cho.get('dcType'))
        has_medium = bool(aggregated_cho.get('medium'))
        has_subject = bool(aggregated_cho.get('subject'))
        dctype_count += has_dctype
        medium_count += has_medium
        both_count += has_dctype and has_medium
        subject_count += has_subject

    dctype_percentage = (dctype_count / total_records) * 100 if total_records > 0 else 0
    medium_percentage = (medium_count / total_records) * 100 if total_records > 0 else 0
    both_percentage = (both_count / total_records) * 100 if total_records > 0 else 0
    subject_percentage = (subject_count / total_records) * 100 if total_records > 0 else 0
    dctype_only_count = dctype_count - both_count
    medium_only_count = medium_count - both_count
    neither_count = total_records - dctype_only_count - medium_only_count - both_count


    return {
        'total_records': total_records,
        'dctype_only_count': dctype_only_count,
        'medium_only_count': medium_only_count,
        'both_count': both_count,
        'neither_count': neither_count,
        'dctype_count': dctype_count,
        'medium_count': medium_count,
        'subject_count': subject_count,
        'invalid_json_lines': invalid_json_lines,
        'dctype_percentage': dctype_percentage,
        'medium_percentage': medium_percentage,
        'both_percentage': both_percentage,
        'subject_percentage': subject_percentage
    }

In [56]:
# iterate over jsonl files 
def iter_jsonl_records(file):
    with file.open('r', encoding='utf-8') as input_file:
        for line_number, line in enumerate(input_file, start=1):
            if not line.strip():
                continue
            try:
                yield json.loads(line)
            except json.JSONDecodeError as error:
                print(f"Skipping invalid JSON in {file.name}, line {line_number}: {error}")
                yield None

In [57]:
# Overall statistics across all records#

jsonl_files = sorted(Path(dst_path).rglob('*.jsonl'))
if not jsonl_files:
    raise FileNotFoundError(f"No JSONL files found in {dst_path!r}")

all_records = (
    record
    for file in jsonl_files
    for record in iter_jsonl_records(file)
)

overall_stats = calculate_percentages(all_records)

overall_summary = {
    'Total valid records': overall_stats['total_records'],
    'Records with dcType': f"{overall_stats['dctype_count']:,} ({overall_stats['dctype_percentage']:.2f}%)",
    'Records with medium': f"{overall_stats['medium_count']:,} ({overall_stats['medium_percentage']:.2f}%)",
    'Records with both': f"{overall_stats['both_count']:,} ({overall_stats['both_percentage']:.2f}%)",
    'Records with subject': f"{overall_stats['subject_count']:,} ({overall_stats['subject_percentage']:.2f}%)",
    'dcType only': overall_stats['dctype_only_count'],
    'Medium only': overall_stats['medium_only_count'],
    'Neither field': overall_stats['neither_count'],
    'Invalid JSON lines skipped': overall_stats['invalid_json_lines'],
}
overall_df = pd.DataFrame.from_dict(overall_summary, orient='index', columns=['All records'])
display(overall_df)

,All records
Total valid records,2636673
Records with dcType,"2,444,252 (92.70%)"
Records with medium,"809,780 (30.71%)"
Records with both,"722,865 (27.42%)"
Records with subject,"1,360,391 (51.59%)"
dcType only,1721387
Medium only,86915
Neither field,105506
Invalid JSON lines skipped,0


In [58]:
# institution statistics

institution_stats = {}
for file in jsonl_files:
    institution_stats[file.stem] = calculate_percentages(iter_jsonl_records(file))

stats_path = Path('institution_stats.json')
with stats_path.open('w', encoding='utf-8') as output_file:
    json.dump(institution_stats, output_file, indent=2)

with stats_path.open('r', encoding='utf-8') as input_file:
    saved_institution_stats = json.load(input_file)

institution_df = pd.DataFrame.from_dict(saved_institution_stats, orient='index')
institution_df.index.name = 'institution'
institution_df = institution_df.rename(columns={
    'dctype_only_count': 'dcType only',
    'medium_only_count': 'medium only',
    'both_count': 'both',
    'neither_count': 'neither',
    'invalid_json_lines': 'invalid JSON lines',
    'dctype_percentage': 'dcType (%)',
    'medium_percentage': 'medium (%)',
    'both_percentage': 'both (%)',
    'subject_count': 'subject count',
    'subject_percentage': 'subject (%)',
})
percentage_columns = ['dcType (%)', 'medium (%)', 'both (%)', 'subject (%)']
institution_df[percentage_columns] = institution_df[percentage_columns].round(2)
display(institution_df)

,total_records,dcType only,medium only,both,neither,dctype_count,medium_count,subject count,invalid JSON lines,dcType (%),medium (%),both (%),subject (%)
institution,,,,,,,,,,,,,
akademie-der-bildenden-kuenste-core__2026-01-14T11-50-22Z,9043,1,0,9042,0,9043,9042,0,0,100.00,99.99,99.99,0.00
alban-berg__2025-06-17T08-09-23.305000Z,1701,1671,0,30,0,1701,30,0,0,100.00,1.76,1.76,0.00
albertina__2026-07-13T14-03-47Z,137747,569,0,137178,0,137747,137178,0,0,100.00,99.59,99.59,0.00
arbeiterzeitung-archiv__2026-09-16T07-29-47Z,14826,14826,0,0,0,14826,0,0,0,100.00,0.00,0.00,0.00
ars-electronica__2026-06-17T10-13-21Z,10643,10643,0,0,0,10643,0,10624,0,100.00,0.00,0.00,99.82
...,...,...,...,...,...,...,...,...,...,...,...,...,...
vorau-klosterneuburg-repos__2026-10-13T12-46-52Z,236,236,0,0,0,236,0,235,0,100.00,0.00,0.00,99.58
westlicht-core__2025-09-18T11-01-03Z,105,105,0,0,0,105,0,105,0,100.00,0.00,0.00,100.00
wien-bibliothek__2025-05-16T11-55-31Z,168157,139471,0,28686,0,168157,28686,148362,0,100.00,17.06,17.06,88.23


In [59]:
# Collect Kulturpool IDs for records with neither dcType nor medium.
def collect_neither_ids(records):
    neither_ids = []

    for record in records:
        if record is None:
            continue

        aggregated_cho = record.get('content', {}).get('aggregatedCHO', {})
        if not aggregated_cho.get('dcType') and not aggregated_cho.get('medium'):
            record_id = record.get('kp_id')
            if not record_id:
                raise ValueError('A record missing both fields has no kp_id')
            neither_ids.append(record_id)

    return neither_ids


neither_ids = collect_neither_ids(
    record
    for file in sorted(Path(dst_path).rglob('*.jsonl'))
    for record in iter_jsonl_records(file)
)

neither_ids_path = Path('neither_ids.json')
with neither_ids_path.open('w', encoding='utf-8') as output_file:
    json.dump(neither_ids, output_file, indent=2)